# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Classification, deployed as ranking.** The real business question is *"which pages should an
editor refresh first?"* — a decision. To support it we predict a binary, observed outcome:
`is_declining_label` = did this page's search impressions fall more than 20% in the last 30 days
vs the prior 30 days? That is a **binary classification** task (target: yes/no from a measured
outcome). We then turn the model's probability into a **priority score** so the editor gets a
ranked queue — classification feeds ranking, but the model itself is a classifier.

Why not pure ranking/regression? We have a clean yes/no label, so classification is the honest
fit; we add the ranking layer only at deployment.

In [ ]:
# Section 1 — load the slice and derive the binary label
from pathlib import Path
import numpy as np
import pandas as pd

# Find the anonymized CSV whether we run locally or in Colab.
def find_raw_csv():
    here = Path.cwd()
    for parent in [here, *here.parents]:
        cand = parent / "data" / "raw" / "content_refresh_anonymized.csv"
        if cand.exists():
            return cand
    return Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(find_raw_csv())
df.columns = df.columns.str.strip()

# The label is DERIVED from an observed outcome (the 90d->30d impression trend),
# not from a human opinion. trend_direction is computed upstream; we rebuild it here.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

base_rate = df["is_declining_label"].mean()
print(f"Rows (one content item each): {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Declining (label=1) share: {base_rate:.3f}  -> a balanced binary problem")

# Chosen task type (justified in the markdown above):
# PRIMARY = binary CLASSIFICATION (predict is_declining_label);
# DEPLOYED = a scoring/ranking layer on top, because the real decision is "which pages first?".
print("Task type: binary classification, deployed as a ranked refresh-priority score.")

: 

## 2. Target or proxy

**Target: `is_declining_label` (1 = declining, 0 = not).** It is built from an *observed* outcome:
Google Search Console impression counts over the last 30 days vs the previous 30 days
(`trend_direction == "down"`). This is measured signal, not someone's opinion or a hand-written
rule — so the model learns the real world, not a rule.

**Leakage guard (critical):** the label is computed *from* `trend_direction` and `trend_pct`.
Those two columns must therefore **never** be model features (notebook 02 proves what happens
if you leak them). IDs (`content_id`, `client_id`) are for grouping/splitting only.

In [2]:
# Section 2 — target / proxy: prove provenance + enforce the leakage guard
# Target: is_declining_label (1 = page's impressions fell >20% last 30d vs prev 30d).
# Provenance: OBSERVED outcome from GSC impression counts, NOT a hand-written rule.
# Leakage guard: trend_direction & trend_pct define the label, so they can NEVER be features.

LEAKY = {"trend_direction", "trend_pct"}
candidate_features = [c for c in df.columns
                      if c not in LEAKY
                      and c not in {"content_id", "client_id", "is_declining_label"}]
print(f"Candidate features (IDs + leaky cols excluded): {len(candidate_features)}")
print("Leakage-guarded columns removed from features:", sorted(LEAKY))
print("IDs used for grouping/splitting only, never as features: content_id, client_id")

# Sanity: the label is genuinely binary and observed, no missing values after derivation.
print("Label values:", sorted(df["is_declining_label"].unique()))
print("Label missing values:", int(df["is_declining_label"].isna().sum()))

Candidate features (IDs + leaky cols excluded): 40
Leakage-guarded columns removed from features: ['trend_direction', 'trend_pct']
IDs used for grouping/splitting only, never as features: content_id, client_id
Label values: [np.int64(0), np.int64(1)]
Label missing values: 0


## 3. Success metric

**One metric I can defend: Precision@50 on a client-holdout.** Of the 50 pages the system says
to fix first, what fraction are genuinely declining? Higher = fewer wasted editor hours.

- **Floor to beat:** the reference hand-rule baseline scores **≈ 0.24** Precision@50 on the
  client-holdout test set (the authoritative number from the reference pipeline — confirmed in
  the cell below).
- **Model target:** client-holdout Precision@50 **≈ 0.74** (from `03_train_model` / `04_evaluate`),
  roughly a **3× lift** over the baseline. The stable claim is the lift, not the third decimal.
- **Model-selection metric:** **ROC-AUC** — threshold-free, so it is fair at the ~0.54 base rate
  where raw accuracy would be misleading.

In [3]:
# Section 3 — success metric: the floor we must beat (Precision@50 on the client-holdout)
# The reference pipeline (scripts/01..04) computes the authoritative numbers and saves them to
# outputs/model_results.json. We read those directly when present; if the pipeline has not been
# run yet (e.g. fresh Colab), we fall back to the documented values from README.md / GUIDE.md FAQ.
import json as _json
from pathlib import Path as _Path

_model_results_path = None
for _parent in [Path.cwd(), *Path.cwd().parents]:
    _cand = _parent / "outputs" / "model_results.json"
    if _cand.exists():
        _model_results_path = _cand
        break
if _model_results_path is not None:
    _mr = _json.loads(_model_results_path.read_text())
    baseline_p50 = _mr["baseline"]["baseline_precision_at_50"]
    best_model = _mr["best_model"]["name"]
    model_p50 = _mr["models"][best_model]["precision_at_50"]
    src = "reference pipeline (outputs/model_results.json)"
else:
    # Documented values from README.md / GUIDE.md FAQ, reproduced on the shipped library stack.
    baseline_p50, model_p50, best_model = 0.240, 0.740, "random_forest"
    src = "documented values (README/GUIDE FAQ) — run scripts/run_all.py to recompute"

no_skill_p50 = base_rate  # "flag every page as declining"
lift = model_p50 / baseline_p50 if baseline_p50 else float("nan")

print(f"No-skill Precision@50 (flag all pages):  {no_skill_p50:.3f}  (unbalanced base rate)")
print(f"Baseline rules Precision@50 (floor):     {baseline_p50:.3f}   source: {src}")
print(f"Best model ({best_model}) Precision@50:  {model_p50:.3f}")
print(f"Lift over baseline:                      {lift:.1f}x")
print("Model-selection metric: ROC-AUC (threshold-free, fair at the ~0.54 base rate)")
print("Our success bar: beat the 0.24 baseline floor on client-holdout Precision@50.")

No-skill Precision@50 (flag all pages):  0.542  (unbalanced base rate)
Baseline rules Precision@50 (floor):     0.240   source: reference pipeline (outputs/model_results.json)
Best model (random_forest) Precision@50:  0.740
Lift over baseline:                      3.1x
Model-selection metric: ROC-AUC (threshold-free, fair at the ~0.54 base rate)
Our success bar: beat the 0.24 baseline floor on client-holdout Precision@50.


## 4. The unit of analysis, as a real dataframe

**One row = one content item (page).** 30,000 pseudonymized pages across 32 clients, with
trailing-90-day search/analytics metrics. The cell loads it and shows the shape, the derived
label, and a few rows — confirming the grain before we model on it.

In [4]:
# Section 4 — unit of analysis: one row = one content item (page)
print("One row = ONE content item (page), pseudonymized, 32 clients, trailing-90d metrics.")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} cols (incl. derived label)")
print("\nFirst 3 rows (selected columns):")
cols = ["content_id", "client_id", "impressions_90d", "clicks_90d", "avg_position",
        "days_since_last_update", "trend_direction", "is_declining_label"]
print(df[cols].head(3).to_string(index=False))
print(f"\nClients: {df['client_id'].nunique()}  | content items: {df['content_id'].nunique()}")

One row = ONE content item (page), pseudonymized, 32 clients, trailing-90d metrics.
Shape: 30,000 rows x 45 cols (incl. derived label)

First 3 rows (selected columns):
          content_id         client_id  impressions_90d  clicks_90d  avg_position  days_since_last_update trend_direction  is_declining_label
content_304f48230142 client_f369cb89fc             3803          29          10.6                      20            down                   1
content_a1fb4e703a9e client_4e07408562            15320           7          20.3                      25            down                   1
content_9aa793d4d895 client_7f2253d7e2            12581          11          36.5                      20            down                   1

Clients: 32  | content items: 30000


## 5. Why ML beats a fixed rule here

The decline signal is **too messy for an if-statement**. No single feature separates declining
from non-declining pages (the cell shows the strongest single-feature correlation is weak);
instead the pattern is a tangle of weak, partly non-linear effects — visibility, freshness,
position, engagement, and AI-referral traffic all contribute a little, and they interact. A
model learns that blend from data; a hand rule would need dozens of brittle thresholds that
drift as Google's behavior changes. ML earns its place exactly here.

In [5]:
# Section 5 — why ML beats a fixed rule: show no single feature separates the classes
# Use only leakage-safe numeric candidate features.
feat = [c for c in candidate_features if pd.api.types.is_numeric_dtype(df[c])]
corr = df[feat + ["is_declining_label"]].corr()["is_declining_label"].drop("is_declining_label")
corr = corr.reindex(corr.abs().sort_values(ascending=False).index)

print("Strongest |correlation| of any single numeric feature with 'declining':")
for name, val in corr.head(5).items():
    print(f"  {name:28s} {val:+.3f}")
print(f"\nMax |single-feature correlation| = {corr.abs().max():.3f}")
print("-> No single threshold separates the classes; the pattern is many weak, entangled,")
print("   partly non-linear signals (visibility, freshness, position, engagement, AI traffic).")
print("   A model learns that combination; a hand rule would need dozens of brittle thresholds.")

Strongest |correlation| of any single numeric feature with 'declining':
  days_with_impressions        +0.190
  content_age_days             -0.164
  age_tier_order               -0.156
  impressions_last_30d         -0.094
  word_count                   +0.090

Max |single-feature correlation| = 0.190
-> No single threshold separates the classes; the pattern is many weak, entangled,
   partly non-linear signals (visibility, freshness, position, engagement, AI traffic).
   A model learns that combination; a hand rule would need dozens of brittle thresholds.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.